# DepthScope - Monocular Depth Estimation

This notebook takes a normal 2D photo and predicts depth using the pretrained Depth Anything V2 model.

### How to run:
1. Run the setup and model loading cells.
2. Upload any picture (.jpg or .png).
3. See the generated depth map and download it.


In [ ]:
# install transformers
!pip install -q transformers


In [ ]:
import time
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline
from google.colab import files

# use gpu if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
# load pretrained model
pipe = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=device)


In [ ]:
print("📁 Please upload a photograph (supported formats: JPG, JPEG, PNG)...")

if IN_COLAB:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("❌ No file uploaded! Please re-run this cell and choose a valid image file.")
    uploaded_image_path = list(uploaded.keys())[0]
else:
    import os
    candidates = [f for f in os.listdir(".") if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp"))]
    if candidates:
        uploaded_image_path = candidates[0]
        print(f"Local environment detected: using '{uploaded_image_path}'")
    else:
        raise ValueError("❌ No image file found in current directory. Please upload or place a valid image here.")

# quick extension check
valid_exts = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
if not uploaded_image_path.lower().endswith(valid_exts):
    raise ValueError(f"❌ Unsupported file format for '{uploaded_image_path}'. Supported formats: JPG, JPEG, PNG.")

print(f"🎉 Upload successful: {uploaded_image_path}")


In [ ]:
try:
    # make sure image is standard rgb
    raw_image = Image.open(uploaded_image_path)
    img_rgb = raw_image.convert("RGB")
    
    img_width, img_height = img_rgb.size
    print("🖼️ Image successfully opened!")
    print(f"📐 Dimensions: {img_width} x {img_height} pixels (Width x Height)")
    print(f"🎨 Image Color Mode: {img_rgb.mode}")
except Exception as e:
    raise RuntimeError(f"❌ Could not open or process image '{uploaded_image_path}': {e}")

In [ ]:
start_time = time.perf_counter()

try:
    # run model forward pass
    with torch.no_grad():
        output = pipe(img_rgb)
    end_time = time.perf_counter()
    
    inference_time_ms = (end_time - start_time) * 1000.0
    print(f"⚡ Inference completed in {inference_time_ms:.2f} ms ({end_time - start_time:.4f} seconds)")
    print(f"📊 Original input size: {img_width} x {img_height}")
except Exception as e:
    raise RuntimeError(f"❌ Depth inference failed: {e}")

In [ ]:
# handle tensor or pil output and rescale to 0-1
if "predicted_depth" in output and output["predicted_depth"] is not None:
    raw = output["predicted_depth"].detach().cpu().numpy().squeeze()
    d_span = float(raw.max() - raw.min())
    depth_norm = (raw - raw.min()) / d_span if d_span > 1e-6 else np.zeros_like(raw)
    depth_map_pil = Image.fromarray((depth_norm * 255.0).astype(np.uint8)).resize(img_rgb.size, Image.Resampling.BILINEAR)
else:
    depth_map_pil = output["depth"].convert("L").resize(img_rgb.size, Image.Resampling.BILINEAR)

depth_display = np.array(depth_map_pil, dtype=np.float32) / 255.0

# side by side preview: rgb, raw depth, heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

axes[0].imshow(img_rgb)
axes[0].set_title(f"1. Original Photograph\n({img_width} x {img_height})", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(depth_display, cmap="gray")
axes[1].set_title("2. Grayscale Depth Map\n(Brighter = Closer, Darker = Farther)", fontsize=13, fontweight="bold")
axes[1].axis("off")

im_color = axes[2].imshow(depth_display, cmap="inferno")
axes[2].set_title("3. Colorized Relative Depth Map\n(Inferno Colormap)", fontsize=13, fontweight="bold")
axes[2].axis("off")

cbar = fig.colorbar(im_color, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label("Relative Disparity / Depth (Normalized [0, 1])", fontsize=11)

plt.show()

print("💡 Note: Relative depth values represent normalized disparity (closer vs farther), not physical metres or centimetres.")


In [ ]:
# export depth map and grab download
output_filename = "depthscope_predicted_depth.png"
depth_map_pil.save(output_filename, format="PNG")
print(f"💾 Depth map saved locally as '{output_filename}'")

if IN_COLAB:
    files.download(output_filename)
    print("✅ Download started!")
else:
    print(f"✅ Depth map saved to disk at '{output_filename}' (skipping browser download outside Colab)")


## 🔬 Experiment Analysis & Conceptual Guide

Now that you have seen the depth map produced by Depth Anything V2, let's break down the science behind what the model is actually doing and where monocular vision meets its limits.

---

### 1. What Does the Depth Map Represent?
- In monocular depth models like **Depth Anything V2**, the predicted value for each pixel represents **relative inverse depth (disparity)**.
- In our normalized visualization (`[0, 1]`):
  - **High values (Brighter / Warm yellow in Inferno colormap)** represent objects that are **closer** to the camera.
  - **Low values (Darker / Cool purple/black in Inferno colormap)** represent objects that are **farther** away in the background.
- This produces a dense ordering of scene elements from foreground to background.

---

### 2. Relative Depth vs. Metric Depth
It is critical to distinguish between **relative depth** and **metric depth**:
| Property | Relative Depth (DepthScope) | Metric Depth |
| :--- | :--- | :--- |
| **Units** | Unitless normalized scale (`[0, 1]`) | Physical units (meters, centimeters, feet) |
| **Output** | Affine-invariant / Disparity ordering | True Euclidean distance ($Z$) from camera plane |
| **Requirements** | Single RGB photograph only | Camera focal length, sensor size, baseline, or LiDAR calibration |
| **Scale Ambiguity** | Inherent: a small toy car close up looks identical to a real car far away | Resolved: known scale and physical distance |
| **Use Cases** | Bokeh blur effects, 3D photography, AR relighting, obstacle prioritization | Robotic grasping, autonomous vehicle navigation, industrial metrology |

> **Scale Ambiguity**: From a single 2D photo without camera calibration, it is mathematically impossible to know whether you are viewing a miniature model 30 cm away or a full-scale building 300 meters away. Relative depth models predict the *ordering* and *shape*, but cannot give real-world meter measurements.

---

### 3. Failure Cases: Transparent and Reflective Surfaces
Monocular depth networks are trained on visual texture and diffuse surface reflection assumptions. When these assumptions fail, depth predictions can distort:
- **Glass and Transparent Windows**: Glass surfaces do not reflect diffuse light in the standard way. The model often sees right through the window and predicts the depth of whatever is behind the glass, or gets confused by faint reflections on the surface.
- **Mirrors and Specular Reflections**: A mirror reflects the distant room or objects. Because the model relies on appearance cues, it often predicts depth *inside* the virtual image of the mirror (assigning it far depth) rather than detecting the flat surface of the mirror itself.
- **Polished Floors and Water**: Wet pavement, marble floors, or water bodies produce mirror-like specular highlights that violate standard Lambertian surface assumptions.

---

### 4. General Limitations of Monocular Depth Estimation
1. **Ill-Posed Mathematical Problem**: Projecting a 3D world onto a 2D plane loses the third dimension ($Z$). Multiple distinct 3D shapes can produce the exact same 2D projection.
2. **Thin Structures & Occlusion Boundaries**: Fine details like power lines, hair strands, bicycle spokes, and tree branches may blur or merge into the background.
3. **Lighting & Shadow Dependence**: Deep shadows can be mistaken for distant voids, while strong spotlight glare can be mistaken for nearby high-frequency surfaces.
4. **Out-of-Distribution Scenes**: Highly unusual perspectives (e.g., extreme drone nadir shots, microscopic imagery, or abstract art) may not match the distribution of the training data.

## 📝 Results & Observations Log

Use the template below to record your findings as you experiment with different photographs in DepthScope.

---

### 🧪 Test 1: Indoor Scene (e.g., Desk, Living Room, Hallway)
- **Image Description**:
- **Inference Time (ms)**:
- **Key Observations**:
  - How well were foreground objects (e.g., mug, keyboard, chair) separated from the background?
  - Did edge boundaries appear sharp or blurry?
  - Did indoor ceiling lights or shadows create false depth artifacts?

---

### 🌲 Test 2: Outdoor Scene (e.g., Street, Park, Architecture)
- **Image Description**:
- **Inference Time (ms)**:
- **Key Observations**:
  - How did the model handle the sky? (The sky should have very low disparity / farthest distance).
  - How well did perspective cues (roads, sidewalks, rows of trees) translate into smooth depth gradients?
  - Were distant buildings properly separated from foreground trees or pedestrians?

---

### 🪞 Test 3: Reflective or Transparent Object (e.g., Mirror, Glass Window, Water)
- **Image Description**:
- **Inference Time (ms)**:
- **Key Observations**:
  - Did the model predict the depth of the glass surface or the objects behind the glass?
  - For mirrors, did the depth map depict the wall where the mirror hangs or the reflected room inside?
  - How did specular reflections affect nearby surface predictions?

---

### 📌 Summary of Personal Takeaways
*(Record your reflections on the performance of Depth Anything V2 Small, speed vs. accuracy trade-offs, and where you would apply relative depth maps in real-world creative or technical projects.)*